# LAB | Using MCP in LangChain

This notebook demonstrates how to integrate MCP (Model Context Protocol) servers with LangChain agents — enabling them to access external tools and resources through a standardized, unified interface.

## Learning Objectives

- Integrate MCP servers with LangChain agents
- Use MCP tools in LangChain agent workflows
- Access MCP resources for context in LangChain
- Build a practical agent that leverages MCP capabilities
- Compare MCP integration vs direct API integration approaches

---

## Step 1: Setup and Installation

Install the required packages. We need:
- `langchain` — core framework
- `langchain-openai` — OpenAI LLM integration
- `langchain-mcp-adapters` — bridges MCP servers to LangChain tools
- `langgraph` — agent runtime used by `create_agent`
- `mcp` — base MCP client library
- `python-dotenv` — loads API keys from `.env`

In [ ]:
# Install required packages
# Run this cell once, then restart the kernel if needed
%pip install langchain langchain-openai langchain-mcp-adapters langgraph mcp python-dotenv -q

In [27]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# Load environment variables from .env file
# Expected .env contents:
#   OPENAI_API_KEY=sk-...
load_dotenv()

# Verify the API key is present
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY not found. "
        "Create a .env file in this directory with: OPENAI_API_KEY=sk-..."
    )

print("✅ API key loaded successfully")

# Initialise the LLM we'll use throughout this lab
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print(f"✅ LLM initialised: {llm.model_name}")

✅ API key loaded successfully
✅ LLM initialised: gpt-4o-mini


---

## Step 2: Connect to an MCP Server

We use the **LangChain documentation MCP server** as our example — it's publicly accessible over HTTP and requires no local setup.

| Property | Value |
|---|---|
| Server URL | `https://docs.langchain.com/mcp` |
| Transport | HTTP |
| Provides | LangChain / LangGraph / LangSmith docs |

`MultiServerMCPClient` accepts a dict of named server configs. Each config specifies `transport` and the connection URL.

In [28]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# ------------------------------------------------------------------
# Configure MCP server(s)
# Keys are arbitrary server names; values are connection configs.
# ------------------------------------------------------------------
mcp_client = MultiServerMCPClient({
    "langchain-docs": {
        "transport": "http",
        "url": "https://docs.langchain.com/mcp"
    }
})

print("✅ MCP client configured")
print("   Server : langchain-docs")
print("   URL    : https://docs.langchain.com/mcp")

✅ MCP client configured
   Server : langchain-docs
   URL    : https://docs.langchain.com/mcp


---

## Step 3: Load MCP Tools into LangChain

`get_tools()` fetches all tools exposed by the connected MCP servers and wraps them as standard LangChain tools.

> **Important:** `MultiServerMCPClient` is **stateless by default**. Every call to `get_tools()` opens a fresh session internally and tears it down automatically — no `async with` context manager is required.

In [29]:
# Fetch tools from the MCP server
# This is an async call — Jupyter supports top-level await
mcp_tools = await mcp_client.get_tools()

print(f"✅ Loaded {len(mcp_tools)} tool(s) from MCP server(s)\n")
print("Available tools:")
for tool in mcp_tools:
    description_preview = tool.description[:100] if tool.description else "No description"
    print(f"  • {tool.name}")
    print(f"    {description_preview}...")

✅ Loaded 2 tool(s) from MCP server(s)

Available tools:
  • search_docs_by_lang_chain
    Search across the Docs by LangChain knowledge base to find relevant information, code examples, API ...
  • query_docs_filesystem_docs_by_lang_chain
    Run a read-only shell-like query against a virtualized, in-memory filesystem rooted at `/` that cont...


---

## Step 4: Create a LangChain Agent with MCP Tools

We use `create_agent` from `langchain.agents` — a modern agent that alternates between reasoning and tool use.

Key parameters:
- `model` — the LLM
- `tools` — list of LangChain tools (our MCP tools)
- `prompt` — system message that sets the agent's behaviour

In [30]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# Load tools fresh for this agent
tools = await mcp_client.get_tools()

# Build the agent
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful assistant that answers questions about LangChain, "
        "LangGraph, and LangSmith by querying the official documentation. "
        "Always base your answers on what you find in the documentation."
    )
)

print(f"✅ Agent created")
print(f"   Tools available : {[t.name for t in tools]}")

✅ Agent created
   Tools available : ['search_docs_by_lang_chain', 'query_docs_filesystem_docs_by_lang_chain']


In [31]:
# ---------------------------------------------------------------
# Test 1: Basic documentation query
# ---------------------------------------------------------------
question = "How do I create a LangChain agent with tools?"
print(f"Question: {question}\n")

result = await agent.ainvoke({
    "messages": [HumanMessage(content=question)]
})

print("Answer:")
print(result["messages"][-1].content)

Question: How do I create a LangChain agent with tools?

Answer:
To create a LangChain agent with tools, you can use the `create_agent` function. This function allows you to build agents that combine language models with tools, enabling them to reason about tasks, decide which tools to use, and iteratively work towards solutions.

### Basic Steps to Create an Agent

1. **Import Necessary Modules**:
   You need to import the `create_agent` function and any tools you want to use.

   ```python
   from langchain.agents import create_agent
   ```

2. **Define Your Tools**:
   Tools can be defined as plain Python functions or coroutines. You can use the `@tool` decorator to customize tool names, descriptions, and argument schemas.

   ```python
   from langchain.tools import tool

   @tool
   def my_tool(input):
       # Tool logic here
       return "Result from my_tool"
   ```

3. **Create the Agent**:
   Use the `create_agent` function to instantiate your agent. You can specify a model a

In [34]:
# ---------------------------------------------------------------
# Test 2: LangGraph-specific query
# ---------------------------------------------------------------
question2 = "What is LangGraph and how does it differ from standard LangChain agents?"
print(f"Question: {question2}\n")

result2 = await agent.ainvoke({
    "messages": [HumanMessage(content=question2)]
})

print("Answer:")
print(result2["messages"][-1].content)

Question: What is LangGraph and how does it differ from standard LangChain agents?

Answer:
**LangGraph** is an orchestration framework designed to enhance the capabilities of LangChain by providing a more structured way to build and manage agents. It focuses on the underlying capabilities important for agent orchestration, such as durability, streaming, and tracing. 

### Key Differences Between LangGraph and LangChain Agents:

1. **Orchestration Framework**: 
   - **LangGraph** serves as an orchestration framework that allows for more complex agent interactions and management. It is built to work seamlessly with LangChain components but adds additional layers of functionality for agent orchestration.
   - **LangChain** provides the foundational components for building applications and agents but does not inherently include the orchestration capabilities that LangGraph offers.

2. **Customization and Flexibility**:
   - **LangGraph** allows for deeper customization of agents and their

---

## Step 5: Access MCP Resources

Beyond tools (which execute actions), MCP servers can also expose **resources** — read-only data blobs that provide background context.

Resources are useful for injecting reference material (e.g. a documentation index, a config file, a dataset) into your agent's context without requiring a tool call.

In [35]:
# Fetch available resources from all connected MCP servers
# Returns a list of Blob objects
mcp_resources = await mcp_client.get_resources()

print(f"Resources returned: {len(mcp_resources)}")

if mcp_resources:
    for resource in mcp_resources:
        print(f"\n  Resource: {resource}")
else:
    print(
        "\nℹ️  The LangChain documentation MCP server does not expose explicit \n"
        "   resources — it exposes its content via tools instead. \n"
        "   Servers like a local filesystem MCP server would list files here."
    )

Resources returned: 1

  Resource: metadata={'uri': AnyUrl('mintlify://skills/langchain')} data='---\nname: Langchain\ndescription: Use when building AI agents and applications with LLMs, integrating tools and models, creating multi-agent systems, or deploying agents to production. Reach for LangChain when you need to quickly build agents with tool calling, memory, streaming, and observability.\nmetadata:\n    mintlify-proj: langchain\n    version: "1.0"\n---\n\n# LangChain Skill\n\n## Product summary\n\nLangChain is an open-source framework for building agents and LLM applications. It provides a prebuilt agent architecture, integrations for hundreds of LLMs (OpenAI, Anthropic, Google, etc.), and tools for orchestrating complex workflows. Key files and commands:\n\n- **Core packages**: `langchain` (framework), `langchain-openai`, `langchain-anthropic`, etc. (provider integrations)\n- **Key imports**: `from langchain.agents import create_agent`, `from langchain.tools import tool`, `from

### How to use resources when they are available

When a server does expose resources, you can pull them into your agent's context like this:

```python
# Example (filesystem MCP server)
resources = await mcp_client.get_resources()

# Build context string from resource blobs
context = "\n\n".join(r.data.decode() for r in resources if r.data)

agent_with_context = create_agent(
    model=llm,
    tools=tools,
    system_prompt=f"You have access to the following documents:\n\n{context}\n\nUse them to answer questions."
)
```

---

## Step 6: Tool Interceptors (Middleware)

Tool interceptors let you wrap every MCP tool call with custom logic — logging, validation, retry, caching, rate-limiting, etc.

An interceptor is an **async callable** with the signature:

```python
async def my_interceptor(request, handler):
    # request.name      — tool name
    # request.args      — tool arguments dict
    result = await handler(request)   # call next interceptor / actual tool
    # result.content    — tool output
    return result
```

Multiple interceptors compose in an **onion** pattern — the first in the list is the outermost layer.

In [36]:
import time

# ---------------------------------------------------------------
# Interceptor 1: Logging — records every call and its latency
# ---------------------------------------------------------------
async def logging_interceptor(request, handler):
    """Log tool name, arguments, and execution time."""
    t0 = time.perf_counter()
    print(f"[TOOL CALL] {request.name}")
    print(f"  args : {request.args}")
    result = await handler(request)
    elapsed = time.perf_counter() - t0
    snippet = str(result.content)[:120].replace("\n", " ")
    print(f"  time : {elapsed:.2f}s")
    print(f"  out  : {snippet}...\n")
    return result


# ---------------------------------------------------------------
# Interceptor 2: Call counter — tracks how many times tools fire
# ---------------------------------------------------------------
call_counts: dict[str, int] = {}

async def counter_interceptor(request, handler):
    """Count how many times each tool is invoked."""
    call_counts[request.name] = call_counts.get(request.name, 0) + 1
    return await handler(request)


# Create a new MCP client that uses both interceptors
mcp_client_instrumented = MultiServerMCPClient(
    {
        "langchain-docs": {
            "transport": "http",
            "url": "https://docs.langchain.com/mcp"
        }
    },
    tool_interceptors=[logging_interceptor, counter_interceptor]
)

print("✅ MCP client with logging + counter interceptors configured")

✅ MCP client with logging + counter interceptors configured


In [37]:
# Build an agent using the instrumented client
instrumented_tools = await mcp_client_instrumented.get_tools()

instrumented_agent = create_agent(
    model=llm,
    tools=instrumented_tools,
    system_prompt="You are a helpful LangChain documentation assistant."
)

print("=" * 60)
question = "What is LangChain and what are its main features?"
print(f"Question: {question}")
print("=" * 60 + "\n")

result = await instrumented_agent.ainvoke({
    "messages": [HumanMessage(content=question)]
})

print("=" * 60)
print("\nFinal Answer:")
print(result["messages"][-1].content)
print("\nTool call counts:", call_counts)

Question: What is LangChain and what are its main features?

[TOOL CALL] search_docs_by_lang_chain
  args : {'query': 'What is LangChain and its main features?'}
  time : 2.61s
  out  : [TextContent(type='text', text='Title: LangChain overview\nLink: https://docs.langchain.com/oss/javascript/langchain/ove...


Final Answer:
LangChain is an open-source framework designed to facilitate the development of applications that utilize language models. It provides a structured way to build applications that can interact with various data sources, APIs, and other components, making it easier to create complex workflows involving natural language processing.

### Main Features of LangChain:
1. **Modular Components**: LangChain offers a variety of modular components that can be combined to create custom applications. This includes tools for managing prompts, chains, and agents.

2. **Integration with Data Sources**: It allows seamless integration with different data sources, enabling applications

---

## Step 7: Multi-Server Setup

One of MCP's core strengths is connecting to **multiple servers simultaneously**. Each server contributes its own set of tools, all accessible through a single unified client.

Below we connect to the LangChain docs MCP server alongside Claude Code as an optional local MCP server. In a real project you might also add a filesystem server, a database server, or a custom API server.

In [38]:
from shutil import which

# Multi-server client — add more entries to the dict to include more servers
# Claude Code can act as an MCP server locally if the `claude` CLI is installed.
# It exposes Claude's tools via `claude mcp serve`.
server_configs = {
    "langchain-docs": {
        "transport": "http",
        "url": "https://docs.langchain.com/mcp"
    }
}

if which("claude"):
    server_configs["claude-code"] = {
        "transport": "stdio",
        "command": "claude",
        "args": ["mcp", "serve"]
    }
    print("✅ Claude Code MCP server enabled")
else:
    print("ℹ️ Claude CLI not found, skipping Claude Code MCP server")

multi_mcp_client = MultiServerMCPClient(server_configs)

all_tools = await multi_mcp_client.get_tools()
print(f"✅ Total tools across all servers: {len(all_tools)}")
for t in all_tools:
    print(f"  • {t.name}")

ℹ️ Claude CLI not found, skipping Claude Code MCP server
✅ Total tools across all servers: 2
  • search_docs_by_lang_chain
  • query_docs_filesystem_docs_by_lang_chain


---

## Step 8: Practical Example — Documentation Q&A Agent

Putting it all together: a polished documentation assistant that:

1. Connects to the LangChain MCP server
2. Uses a logging interceptor for observability
3. Handles a short multi-question session

In [39]:
# ---------------------------------------------------------------
# Complete, self-contained documentation assistant
# ---------------------------------------------------------------

session_log: list[dict] = []   # track questions + answers

async def logging_interceptor_v2(request, handler):
    """Minimal logging interceptor for the practical example."""
    print(f"  🔧 Tool called: {request.name}")
    result = await handler(request)
    return result


# Set up client and agent
docs_client = MultiServerMCPClient(
    {"langchain-docs": {"transport": "http", "url": "https://docs.langchain.com/mcp"}},
    tool_interceptors=[logging_interceptor_v2]
)

docs_tools = await docs_client.get_tools()

docs_agent = create_agent(
    model=llm,
    tools=docs_tools,
    system_prompt=(
        "You are an expert LangChain documentation assistant. "
        "Answer questions concisely and accurately using the official docs. "
        "If the docs do not cover a topic, say so clearly."
    )
)

print(f"✅ Documentation agent ready with {len(docs_tools)} tool(s)")

✅ Documentation agent ready with 2 tool(s)


In [40]:
# Run a set of questions through the agent
questions = [
    "What is the difference between a Chain and an Agent in LangChain?",
    "How do I add memory to a LangChain agent?",
    "What streaming options does LangChain support?",
]

for i, q in enumerate(questions, 1):
    print(f"\n{'=' * 65}")
    print(f"Q{i}: {q}")
    print("=" * 65)

    response = await docs_agent.ainvoke(
        {"messages": [HumanMessage(content=q)]}
    )
    answer = response["messages"][-1].content

    print(f"\nA{i}: {answer}")
    session_log.append({"question": q, "answer": answer})

print(f"\n✅ Session complete — {len(session_log)} question(s) answered")


Q1: What is the difference between a Chain and an Agent in LangChain?
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_docs_by_lang_chain
  🔧 Tool called: search_do

---

## Step 9 (Optional): MCP vs Direct API Integration

To understand when to reach for MCP versus a direct API call, let's implement the same documentation query both ways and compare.

In [41]:
# ---------------------------------------------------------------
# Approach A: Direct API integration (no MCP)
# ---------------------------------------------------------------
import httpx
from langchain_core.tools import Tool

def fetch_langchain_docs_direct(query: str) -> str:
    """
    Fetch LangChain documentation by directly calling the
    llms.txt index and doing a simple keyword search.
    No MCP involved — pure HTTP.
    """
    try:
        response = httpx.get(
            "https://docs.langchain.com/llms.txt",
            timeout=10
        )
        lines = response.text.splitlines()
        matches = [l for l in lines if query.lower() in l.lower()]
        if matches:
            return "\n".join(matches[:10])
        return f"No results found for '{query}' in the docs index."
    except Exception as e:
        return f"Error fetching docs: {e}"


# Wrap as a LangChain Tool
direct_docs_tool = Tool(
    name="fetch_langchain_docs_direct",
    func=fetch_langchain_docs_direct,
    description="Search the LangChain documentation index directly (no MCP)."
)

# Build a direct-API agent
direct_agent = create_agent(
    model=llm,
    tools=[direct_docs_tool],
    system_prompt="You are a documentation assistant. Use the provided tool to answer questions."
)

# Test the direct agent
direct_question = "What is LangChain?"
print(f"Direct API — Question: {direct_question}\n")
direct_result = await direct_agent.ainvoke(
    {"messages": [HumanMessage(content=direct_question)]}
)
print("Direct API — Answer:")
print(direct_result["messages"][-1].content)

Direct API — Question: What is LangChain?

Direct API — Answer:
LangChain is a framework designed for developing applications powered by language models. It provides tools and components to facilitate the integration of language models into various applications, enabling developers to build more complex and capable systems that leverage natural language processing. LangChain supports various functionalities, including prompt management, chaining together multiple calls to language models, and integrating with external data sources and APIs. 

If you need more specific information or details about certain features, feel free to ask!


In [42]:
# ---------------------------------------------------------------
# Approach B: MCP integration (same question)
# ---------------------------------------------------------------
mcp_question = "What is LangChain?"
print(f"MCP — Question: {mcp_question}\n")

mcp_result = await docs_agent.ainvoke(
    {"messages": [HumanMessage(content=mcp_question)]}
)
print("MCP — Answer:")
print(mcp_result["messages"][-1].content)

MCP — Question: What is LangChain?

  🔧 Tool called: search_docs_by_lang_chain
MCP — Answer:
LangChain is an open-source framework designed for developing applications powered by language models. It provides tools and components to facilitate the integration of language models into various applications, enabling developers to build complex workflows and functionalities that leverage natural language processing capabilities.

For more details, you can visit the [LangChain overview](https://docs.langchain.com/oss/python/langchain/overview).


### Comparison: MCP vs Direct API

| Dimension | MCP Integration | Direct API Integration |
|---|---|---|
| **Protocol** | Standardised (MCP) | Custom per service |
| **Boilerplate** | Low — adapters handle conversion | Higher — must write wrapper per API |
| **Multi-service** | Easy — add servers to dict | Hard — each service needs its own code |
| **Observability** | Built-in via interceptors | Manual |
| **Performance** | Small overhead from abstraction | Minimal |
| **Control** | Limited to what the server exposes | Full API surface |
| **Best for** | 2+ integrations, cross-framework reuse | Single integration, full API control |

**Rule of thumb:**  
- ≥ 2 external systems → favour MCP  
- 1 system, performance-critical → favour direct API

---

## Key Takeaways

1. **`langchain-mcp-adapters`** converts MCP server tools into native LangChain tools with zero manual wrapping.
2. **`MultiServerMCPClient`** is stateless — no `async with` context manager needed; sessions are managed automatically.
3. **Tool interceptors** follow the `(request, handler) -> result` async pattern and compose like middleware.
4. **Resources** expose read-only data blobs useful for injecting background context into agent prompts.
5. **Multiple servers** can be registered in one client dict, giving the agent access to all their tools simultaneously.
6. MCP shines when you need a standardised, reusable integration layer across multiple services or frameworks.

## Next Steps

- Explore the [MCP Servers Repository](https://github.com/modelcontextprotocol/servers) for ready-made servers (filesystem, GitHub, Slack, databases, …)
- Build a local filesystem MCP server and use it for document analysis
- Combine MCP tools with RAG for retrieval-augmented agents
- Add caching or rate-limiting interceptors for production workloads